# Stage 1b: ML C2 and C3 for pipeline P1, both C1 seeds

Runs `mlpipe.run_stage1b` for C1 seed 0 and then seed 1: three training settings (isolated,
old-pipeline, new-pipeline), C2 AutoBot-Ego style (target-centred), C3 PlanT style in two
variants (no ego input, and ego past path), Boston validation split with early stopping,
augmentation, baselines stored with every result. Models and results are written to
`/kaggle/working/E1_new/{models,results}/stage1b/` for download and reuse.

In [ ]:
import os, sys, glob, json, time, tarfile, subprocess
from pathlib import Path
T0 = time.time()
def clock(m): print(f'[{time.time()-T0:7.1f}s] {m}', flush=True)
# Kaggle unpacks uploaded .tar files: accept the unpacked folder or the .tar itself
import shutil
ROOT = Path('/kaggle/working/E1_new'); ROOT.mkdir(parents=True, exist_ok=True)
for dp, dn, fn in os.walk('/kaggle/input'):
    if 'labsd-e1-mlpipe' in dp: print('  input:', dp, sorted(fn)[:6])
RUN = glob.glob('/kaggle/input/**/mlpipe/run_stage1b.py', recursive=True)
PKG = glob.glob('/kaggle/input/**/e1_new_pkg.tar', recursive=True)
if RUN:
    src = Path(RUN[0]).parent.parent
    shutil.copytree(src / 'mlpipe', ROOT / 'mlpipe', dirs_exist_ok=True)
    data = glob.glob(str(src) + '/**/boston_c23.json', recursive=True) or glob.glob('/kaggle/input/**/boston_c23.json', recursive=True)
    (ROOT / 'data' / 'stage1').mkdir(parents=True, exist_ok=True)
    for f in ['boston_c23.json', 'singapore_test.json']:
        shutil.copy2(Path(data[0]).parent / f, ROOT / 'data' / 'stage1' / f)
elif PKG:
    with tarfile.open(PKG[0]) as tf: tf.extractall(ROOT)
else:
    raise AssertionError('package not found: attach dataset ifty1011/labsd-e1-mlpipe')
assert (ROOT / 'mlpipe' / 'run_stage1b.py').exists() and (ROOT / 'data' / 'stage1' / 'singapore_test.json').exists()
clock('package ready in ' + str(ROOT))
import torch, scipy, numpy
clock(f'python {sys.version.split()[0]} torch {torch.__version__} scipy {scipy.__version__} numpy {numpy.__version__}')
assert torch.cuda.is_available(), 'no GPU'
clock(f'GPU {torch.cuda.get_device_name(0)}')

In [ ]:
def run(args):
    t = time.time()
    p = subprocess.Popen([sys.executable, '-u', '-m'] + args, cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' in line or 'warnings.warn' in line: continue
        print(line, end='', flush=True)
    assert p.wait() == 0, f'{args} failed'
    clock(f'{args} finished in {time.time()-t:.0f}s')
run(['mlpipe.selftest'])

In [ ]:
run(['mlpipe.run_stage1b', '--c1-seed', '0'])

In [ ]:
run(['mlpipe.run_stage1b', '--c1-seed', '1'])

In [ ]:
import shutil
shutil.rmtree(ROOT / 'data', ignore_errors=True)
for f in sorted((ROOT / 'models' / 'stage1b').glob('*')) + sorted((ROOT / 'results' / 'stage1b').glob('*')):
    print(f.relative_to(ROOT), f.stat().st_size)
clock('=== STAGE 1b DONE ===')